# Módulo 52: Pruebas de hipótesis en regresión

Autor: Juliho Castillo Colmenares

> Lección complementaria del ejercicio [m52-pruebas-de-hipotesis-solucion.ipynb](m52-pruebas-de-hipotesis-solucion.ipynb). No es una tarea entregada del programa.

Un coeficiente estimado nunca es exactamente cero, aunque la variable no tenga ningún efecto: siempre hay ruido. Las **pruebas de hipótesis** permiten decidir si un efecto observado es demasiado grande para atribuirlo al azar. En esta lección se construyen a mano las pruebas que reporta `statsmodels` y se aprende a leer su resumen completo:

1. La lógica de una prueba de hipótesis
2. Errores tipo I y tipo II
3. Prueba t para un coeficiente
4. Intervalos de confianza
5. Prueba F global
6. Leer el `summary()` de statsmodels
7. Selección hacia adelante por p-valor
8. Errores frecuentes
9. Ejercicio

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

## 1. La lógica de una prueba de hipótesis

Una prueba enfrenta dos afirmaciones sobre la población:

- $H_0$, la **hipótesis nula**: no hay efecto (por ejemplo, $\beta_j = 0$).
- $H_1$, la **hipótesis alternativa**: sí lo hay ($\beta_j \ne 0$).

Se calcula un **estadístico** cuya distribución es conocida si $H_0$ es cierta. El **p-valor** es la probabilidad, suponiendo $H_0$, de obtener un estadístico al menos tan extremo como el observado. Si es menor que el **nivel de significancia** $\alpha$ (habitualmente 0.05), se rechaza $H_0$.

El p-valor **no** es la probabilidad de que $H_0$ sea cierta: es una medida de qué tan sorprendentes serían los datos si lo fuera.

## 2. Errores tipo I y tipo II

| | $H_0$ cierta | $H_0$ falsa |
| --- | --- | --- |
| Se rechaza $H_0$ | **Error tipo I** (probabilidad $\alpha$) | Decisión correcta (**potencia**, $1-\beta$) |
| No se rechaza $H_0$ | Decisión correcta | **Error tipo II** (probabilidad $\beta$) |

(Aquí $\beta$ es la notación clásica de la probabilidad de error tipo II, no un coeficiente.) Se comprueba por simulación: si el efecto real es cero, la prueba rechaza alrededor del 5 % de las veces; si existe un efecto pequeño, la potencia crece con el tamaño de muestra.

In [2]:
rng = np.random.default_rng(42)

def tasa_rechazo(efecto, n, repeticiones=500, alpha=0.05):
    rechazos = 0
    for _ in range(repeticiones):
        x = rng.normal(size=n)
        y = 1 + efecto * x + rng.normal(size=n)
        p = sm.OLS(y, sm.add_constant(x)).fit().pvalues[1]
        rechazos += p < alpha
    return rechazos / repeticiones

print(f"Sin efecto real (n=50): rechaza {tasa_rechazo(0.0, 50):.3f}  <- error tipo I ≈ α")
for n in [20, 50, 200]:
    print(f"Efecto 0.3, n={n:>3}: potencia ≈ {tasa_rechazo(0.3, n):.3f}")

Sin efecto real (n=50): rechaza 0.058  <- error tipo I ≈ α
Efecto 0.3, n= 20: potencia ≈ 0.236
Efecto 0.3, n= 50: potencia ≈ 0.502


Efecto 0.3, n=200: potencia ≈ 0.992


## 3. Prueba t para un coeficiente

Con $n$ observaciones y $p$ predictores más la constante, el estimador de la varianza del error es

$$\hat\sigma^2 = \frac{SS_{res}}{n - p - 1}, \qquad \widehat{\operatorname{Var}}(\hat\beta) = \hat\sigma^2 (X^\top X)^{-1}.$$

El **error estándar** de $\hat\beta_j$ es la raíz del elemento $j$ de la diagonal, $SE(\hat\beta_j) = \sqrt{\hat\sigma^2 [(X^\top X)^{-1}]_{jj}}$. Para $H_0: \beta_j = 0$:

$$t = \frac{\hat\beta_j}{SE(\hat\beta_j)} \sim t_{n-p-1} \text{ bajo } H_0, \qquad p\text{-valor} = 2\,P(T_{n-p-1} > \lvert t \rvert).$$

Se usa una **submuestra de 2.000 casas** de `kc-house-data.csv` (el archivo tiene 21.613).

In [3]:
casas = pd.read_csv("kc-house-data.csv").sample(n=2000, random_state=42).reset_index(drop=True)
predictores = ["sqft_living", "grade", "view", "bedrooms", "yr_built", "condition"]

casas_tr, casas_te = train_test_split(casas, test_size=0.3, random_state=42)
X = sm.add_constant(casas_tr[predictores])
y = casas_tr["price"]
modelo = sm.OLS(y, X).fit()

# Cálculo a mano
n, k = X.shape                      # k = p + 1 columnas (incluye la constante)
Xv, yv = X.to_numpy(), y.to_numpy()
XtX_inv = np.linalg.inv(Xv.T @ Xv)
beta = XtX_inv @ Xv.T @ yv
residuos = yv - Xv @ beta
sigma2 = residuos @ residuos / (n - k)
se = np.sqrt(sigma2 * np.diag(XtX_inv))
t = beta / se
p = 2 * stats.t.sf(np.abs(t), df=n - k)

a_mano = pd.DataFrame({"coef": beta, "SE": se, "t": t, "p": p}, index=X.columns)
print(a_mano.round(4))
print("\n¿Coincide con statsmodels?",
      np.allclose(se, modelo.bse), np.allclose(t, modelo.tvalues), np.allclose(p, modelo.pvalues))

                     coef           SE        t       p
const        5.147395e+06  505645.4855  10.1799  0.0000
sqft_living  1.496571e+02      12.2889  12.1782  0.0000
grade        1.569388e+05    8689.1328  18.0615  0.0000
view         7.411012e+04    8797.9081   8.4236  0.0000
bedrooms    -2.507412e+04    8554.3012  -2.9312  0.0034
yr_built    -3.134870e+03     258.6025 -12.1223  0.0000
condition    3.734233e+04   10578.3475   3.5301  0.0004

¿Coincide con statsmodels? True True True


Aquí sí se invierte $X^\top X$, porque las columnas no son colineales; de todos modos, `statsmodels` usa una pseudoinversa internamente.

## 4. Intervalos de confianza

El intervalo de confianza al $(1-\alpha)$ para cada coeficiente es

$$\hat\beta_j \pm t_{1-\alpha/2,\; n-p-1}\; SE(\hat\beta_j).$$

Prueba e intervalo son dos caras de lo mismo: $H_0: \beta_j = 0$ se rechaza al nivel $\alpha$ **si y solo si** el 0 queda fuera del intervalo al $(1-\alpha)$.

In [4]:
t_crit = stats.t.ppf(0.975, df=n - k)
ic = pd.DataFrame({"inferior": beta - t_crit * se, "superior": beta + t_crit * se}, index=X.columns)
print(ic.round(2))
print("\n¿Coincide con conf_int()?", np.allclose(ic, modelo.conf_int(alpha=0.05)))
print("¿El 0 queda fuera ⇔ p < 0.05?",
      ((ic["inferior"] > 0) | (ic["superior"] < 0)).equals(modelo.pvalues < 0.05))

               inferior    superior
const        4155486.64  6139304.21
sqft_living      125.55      173.76
grade         139893.64   173984.03
view           56851.54    91368.70
bedrooms      -41854.82    -8293.41
yr_built       -3642.16    -2627.58
condition      16591.12    58093.54

¿Coincide con conf_int()? True
¿El 0 queda fuera ⇔ p < 0.05? True


## 5. Prueba F global

La prueba F contrasta si **algún** predictor sirve, frente a un modelo con solo la constante:

$$H_0: \beta_1 = \dots = \beta_p = 0, \qquad F = \frac{(SS_{tot} - SS_{res})/p}{SS_{res}/(n-p-1)} = \frac{R^2/p}{(1-R^2)/(n-p-1)} \sim F_{p,\; n-p-1}.$$

Un F grande con p-valor pequeño indica que el modelo, en conjunto, explica más que la media.

In [5]:
p_pred = k - 1
r2 = modelo.rsquared
F = (r2 / p_pred) / ((1 - r2) / (n - p_pred - 1))
p_F = stats.f.sf(F, p_pred, n - p_pred - 1)
print(f"F a mano: {F:.2f}   statsmodels: {modelo.fvalue:.2f}")
print(f"p-valor de F: {p_F:.3e}   statsmodels: {modelo.f_pvalue:.3e}")

F a mano: 366.26   statsmodels: 366.26
p-valor de F: 3.569e-282   statsmodels: 3.569e-282


## 6. Leer el `summary()` de statsmodels

In [6]:
print(modelo.summary())

                            OLS Regression Results                            
Dep. Variable:                  price   R-squared:                       0.612
Model:                            OLS   Adj. R-squared:                  0.610
Method:                 Least Squares   F-statistic:                     366.3
Date:                Sat, 17 Oct 2026   Prob (F-statistic):          3.57e-282
Time:                        22:54:38   Log-Likelihood:                -19279.
No. Observations:                1400   AIC:                         3.857e+04
Df Residuals:                    1393   BIC:                         3.861e+04
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
const        5.147e+06   5.06e+05     10.180      

| Elemento | Qué mide | Cómo leerlo |
| --- | --- | --- |
| R-squared | $R^2 = 1 - SS_{res}/SS_{tot}$ | Proporción de varianza explicada (en entrenamiento) |
| Adj. R-squared | $1 - (1 - R^2)\frac{n-1}{n-p-1}$ | Penaliza añadir predictores que aportan poco |
| F-statistic, Prob (F) | Prueba F global | p pequeño: el modelo explica más que la media |
| Log-Likelihood | $\ln L$ del modelo normal | Base de AIC y BIC |
| AIC, BIC | $2k - 2\ln L$ y $k\ln n - 2\ln L$ ($k$ parámetros) | Menor es mejor; solo comparan modelos con la misma $y$ |
| coef, std err, t, P>\|t\| | Prueba t de cada coeficiente | p < α: coeficiente distinto de 0 |
| [0.025, 0.975] | Intervalo de confianza al 95 % | Si no contiene 0, el coeficiente es significativo |
| Omnibus, Jarque-Bera | Normalidad de los residuos | p pequeño: residuos no normales |
| Skew, Kurtosis | Asimetría y curtosis de los residuos | 0 y 3 para una normal |
| Durbin-Watson | Autocorrelación de los residuos | Cerca de 2: sin autocorrelación |
| Cond. No. | Número de condición de $X$ | Muy grande: multicolinealidad o escalas muy distintas |

In [7]:
r2_aj = 1 - (1 - r2) * (n - 1) / (n - p_pred - 1)
aic = 2 * k - 2 * modelo.llf
bic = k * np.log(n) - 2 * modelo.llf
print(f"R² ajustado a mano: {r2_aj:.4f}   statsmodels: {modelo.rsquared_adj:.4f}")
print(f"AIC a mano: {aic:.1f}   statsmodels: {modelo.aic:.1f}")
print(f"BIC a mano: {bic:.1f}   statsmodels: {modelo.bic:.1f}")

R² ajustado a mano: 0.6104   statsmodels: 0.6104
AIC a mano: 38571.2   statsmodels: 38571.2
BIC a mano: 38607.9   statsmodels: 38607.9


En este modelo todos los predictores son significativos al 5 %; el de p-valor más alto es `bedrooms` (0.003). Los residuos están lejos de la normalidad (Jarque-Bera enorme, curtosis cercana a 22), algo típico de precios con colas largas, y Durbin-Watson (1.89) no indica autocorrelación. El número de condición es grande sobre todo por las escalas: `yr_built` vale alrededor de 2000 con poca variación relativa, así que es casi proporcional a la columna de unos.

## 7. Selección hacia adelante por p-valor

La **selección hacia adelante** parte de un modelo con solo la constante y, en cada paso, agrega el candidato con menor p-valor, mientras ese p-valor sea menor que $\alpha$. La constante **no es candidata**: está en todos los modelos.

In [8]:
def seleccion_adelante(datos, respuesta, candidatos, alpha=0.05):
    elegidos = []
    restantes = sorted(candidatos)          # orden fijo: resultado reproducible
    while restantes:
        pvalores = pd.Series({
            c: sm.OLS(respuesta, sm.add_constant(datos[elegidos + [c]])).fit().pvalues[c]
            for c in restantes})
        mejor = pvalores.idxmin()
        if pvalores[mejor] >= alpha:
            break
        elegidos.append(mejor)
        restantes.remove(mejor)
    return elegidos

candidatos = ["sqft_living", "grade", "view", "bedrooms", "bathrooms", "yr_built",
              "condition", "floors", "sqft_lot", "waterfront", "lat", "long"]
elegidos = seleccion_adelante(casas_tr, casas_tr["price"], candidatos)
print("Variables elegidas, en orden:", elegidos)

final = sm.OLS(casas_tr["price"], sm.add_constant(casas_tr[elegidos])).fit()
pred_te = final.predict(sm.add_constant(casas_te[elegidos]))
print(f"R² entrenamiento: {final.rsquared:.3f}")
print(f"R² prueba:        {r2_score(casas_te['price'], pred_te):.3f}")

Variables elegidas, en orden: ['grade', 'waterfront', 'yr_built', 'sqft_living', 'lat', 'view', 'condition', 'long', 'bedrooms', 'bathrooms']


R² entrenamiento: 0.693
R² prueba:        0.627


El $R^2$ que importa es el de **prueba**: mide cómo predice el modelo casas que no participaron ni en el ajuste ni en la selección.

La selección por p-valor tiene riesgos:

- **Comparaciones múltiples.** En cada paso se hacen muchas pruebas; con 20 candidatos sin ningún efecto, se espera que alrededor de uno salga "significativo" al 5 % por azar.
- **p-valores optimistas.** Los p-valores del modelo final no tienen en cuenta que las variables se eligieron mirando esos mismos datos.
- **Inestabilidad.** Con predictores correlacionados, pequeños cambios en los datos cambian las variables elegidas.

In [9]:
rng = np.random.default_rng(42)
ruido = pd.DataFrame(rng.normal(size=(300, 20)), columns=[f"x{i:02d}" for i in range(20)])
y_ruido = rng.normal(size=300)        # y no depende de ninguna x

pv = sm.OLS(y_ruido, sm.add_constant(ruido)).fit().pvalues.drop("const")
print("Variables 'significativas' sin efecto real:", sorted(pv[pv < 0.05].index))
print("Selección hacia adelante sobre ruido:", seleccion_adelante(ruido, y_ruido, list(ruido.columns)))

Variables 'significativas' sin efecto real: ['x14']
Selección hacia adelante sobre ruido: ['x14']


## 8. Errores frecuentes

**Dejar la constante fuera del modelo.** Si la constante entra en la lista de candidatos, puede quedar fuera. Sin constante, `statsmodels` calcula un $R^2$ **no centrado** (compara con 0 en vez de con la media), que sale artificialmente alto y no es comparable:

In [10]:
sin_const = sm.OLS(casas_tr["price"], casas_tr[["sqft_living", "grade"]]).fit()
con_const = sm.OLS(casas_tr["price"], sm.add_constant(casas_tr[["sqft_living", "grade"]])).fit()
print(f"R² sin constante (no centrado): {sin_const.rsquared:.3f}")
print(f"R² con constante:               {con_const.rsquared:.3f}")

R² sin constante (no centrado): 0.829
R² con constante:               0.517


**Reportar el $R^2$ de entrenamiento como validación.** El $R^2$ de `modelo.rsquared` siempre es el de los datos de ajuste. Para validar hay que predecir el conjunto de prueba y calcular `r2_score(y_prueba, predicciones)`, como en la sección 7.

**Leer "p > 0.05" como "no hay efecto".** No rechazar $H_0$ significa que los datos no bastan para descartar el cero, no que el efecto sea cero. Con poca muestra (baja potencia, sección 2), efectos reales pasan desapercibidos.

**Confundir significancia con importancia.** Con muchos datos, efectos minúsculos salen significativos. El tamaño del coeficiente y su intervalo dicen si el efecto importa en la práctica.

## 9. Ejercicio

Simula los salarios mensuales de 400 personas. El salario depende de los años de experiencia y de educación, mientras que 8 variables más son ruido:

```python
rng = np.random.default_rng(42)
n = 400
personas = pd.DataFrame(rng.normal(size=(n, 8)), columns=[f"z{i}" for i in range(8)])
personas["experiencia"] = rng.uniform(0, 30, n)
personas["educacion"] = rng.integers(9, 21, n)
personas["salario"] = 5000 + 300 * personas["experiencia"] + 800 * personas["educacion"] \
                      + rng.normal(0, 4000, n)
```

1. Divide en entrenamiento (70 %) y prueba (30 %) y ajusta el modelo con las 10 variables.
2. ¿Qué coeficientes son significativos al 5 %? Comprueba con sus intervalos de confianza.
3. Calcula a mano el estadístico F global a partir de $R^2$ y compáralo con `fvalue`.
4. Aplica la selección hacia adelante (sin la constante como candidata). ¿Entra alguna variable de ruido?
5. Compara el $R^2$ de entrenamiento y el de prueba del modelo seleccionado.

### Solución

In [11]:
rng = np.random.default_rng(42)
n_p = 400
personas = pd.DataFrame(rng.normal(size=(n_p, 8)), columns=[f"z{i}" for i in range(8)])
personas["experiencia"] = rng.uniform(0, 30, n_p)
personas["educacion"] = rng.integers(9, 21, n_p)
personas["salario"] = 5000 + 300 * personas["experiencia"] + 800 * personas["educacion"] \
                      + rng.normal(0, 4000, n_p)

# 1. División y modelo completo
variables = [f"z{i}" for i in range(8)] + ["experiencia", "educacion"]
p_tr, p_te = train_test_split(personas, test_size=0.3, random_state=42)
completo = sm.OLS(p_tr["salario"], sm.add_constant(p_tr[variables])).fit()

# 2. Significancia e intervalos
tabla = pd.concat([completo.params.rename("coef"), completo.pvalues.rename("p"),
                   completo.conf_int().rename(columns={0: "ic_inf", 1: "ic_sup"})], axis=1)
print(tabla.round(3))
print("Significativas:", sorted(completo.pvalues.drop("const").index[completo.pvalues.drop("const") < 0.05]))

                 coef      p    ic_inf    ic_sup
const        5973.857  0.000  3860.614  8087.100
z0           -294.308  0.246  -792.160   203.543
z1           -309.144  0.189  -771.132   152.844
z2            136.707  0.539  -301.143   574.558
z3            357.101  0.137  -114.182   828.385
z4            128.285  0.586  -335.413   591.983
z5            156.512  0.492  -291.233   604.258
z6            376.291  0.105   -78.623   831.206
z7           -215.930  0.352  -671.760   239.900
experiencia   300.037  0.000   246.546   353.529
educacion     710.181  0.000   575.016   845.346
Significativas: ['educacion', 'experiencia']


In [12]:
# 3. F global a mano
n_tr, p_vars = len(p_tr), len(variables)
F_mano = (completo.rsquared / p_vars) / ((1 - completo.rsquared) / (n_tr - p_vars - 1))
print(f"F a mano: {F_mano:.2f}   statsmodels: {completo.fvalue:.2f}")

# 4. Selección hacia adelante
sel = seleccion_adelante(p_tr, p_tr["salario"], variables)
print("Seleccionadas:", sel)

# 5. Entrenamiento frente a prueba
m_sel = sm.OLS(p_tr["salario"], sm.add_constant(p_tr[sel])).fit()
pred = m_sel.predict(sm.add_constant(p_te[sel]))
print(f"R² entrenamiento: {m_sel.rsquared:.3f}   R² prueba: {r2_score(p_te['salario'], pred):.3f}")

F a mano: 26.99   statsmodels: 26.99
Seleccionadas: ['experiencia', 'educacion']
R² entrenamiento: 0.482   R² prueba: 0.349


Solo experiencia y educación son significativas, y la selección hacia adelante no deja entrar ninguna variable de ruido. Con los valores de la simulación, el $R^2$ poblacional es cercano a 0.47 (la varianza de la parte explicada es parecida a la del ruido); el de prueba sale más bajo que el de entrenamiento por la variabilidad de una muestra de solo 120 personas.

## Resumen

- Una prueba compara $H_0$ con $H_1$ mediante un estadístico y su p-valor; con $\alpha = 0.05$ se acepta un 5 % de errores tipo I, y la potencia crece con el tamaño de muestra.
- La prueba t de cada coeficiente usa $t = \hat\beta_j / SE(\hat\beta_j)$, con $SE$ tomado de $\hat\sigma^2(X^\top X)^{-1}$; el intervalo de confianza excluye el 0 exactamente cuando $p < \alpha$.
- La prueba F global contrasta si algún predictor sirve; el $R^2$ ajustado, el AIC y el BIC penalizan la complejidad.
- El `summary()` reúne además diagnósticos de normalidad (Omnibus, Jarque-Bera), autocorrelación (Durbin-Watson) y condicionamiento.
- La selección hacia adelante nunca trata la constante como candidata, sufre de comparaciones múltiples y se valida con el $R^2$ de prueba.

En el **Módulo 53** se construyen modelos de regresión completos: eliminación hacia atrás, intervalos de predicción y verificación de supuestos.